<div style="background:#1a1a2e;color:#eee;padding:24px 32px;border-radius:10px;font-family:monospace">
<div style="font-size:11px;color:#aaa;letter-spacing:2px;text-transform:uppercase">Senior GenAI Engineer Programme · Section 1 · Week 1 · Module 3</div>
<div style="font-size:24px;font-weight:700;margin:8px 0 4px">Functions, Scoping, Closures &amp; Packages</div>
<div style="font-size:13px;color:#ccc">Tier T0 — Platform Engineer &nbsp;|&nbsp; Anthrolytics &nbsp;|&nbsp; ~5 hours</div>
<div style="font-size:12px;color:#888;margin-top:8px">Prerequisites: Module 01_01_01 (mental model) · Module 01_01_02 (generators)</div>
</div>

## How to use this notebook

1. **Read** the concept. Understand the mechanism before running anything.
2. **Write your prediction** in every `> My prediction:` cell — actually edit the text.
3. **Run the cell.** Compare with your prediction.
4. **Read the sticky phrase.** Say it once.

Hints (`<details>`) are there. Use them only after a genuine attempt.

The reference card at the end is the thing you take into week 2.

## The brief

**Slack — Marcus Okafor (senior engineer) → you, Wednesday 10:22 AM**

> "Hey — Priya says you nailed the mutable default bugs and the streaming reader. Good start. Your next task is one I've seen trip up every engineer who comes from JavaScript: closures. You're going to build a rate limiter and a retry wrapper — both closure-based — that we'll use to wrap every LLM API call in the prototype. Then you're going to package all of it properly so I can `pip install` it from the repo. Demo at 4 PM. If the package installs cleanly and the rate limiter holds up under a 1000-call stress test, I'll mark week 1 ready for Priya's checkpoint review."

---

**What you are building by 4 PM:**
- `utils/rate_limiter.py` — `make_token_bucket()`: closure-based token bucket rate limiter
- `utils/retry.py` — `make_retry()`: closure-based exponential backoff retry wrapper
- The `anthrolytics` package structure with `pyproject.toml`, installable via `uv pip install -e .`

Marcus's closing demo is the final exercise. Score 5/5 on the stress test and the package demo, and week 1 is done.

---
## Part 1 — First-class functions: functions as values

In Python, functions are objects. The same object model from Module 1.1 applies:

```
def process_contract(c):
    return c["contract_id"].upper()

┌───────────────────────────────────────┐
│  function object                      │
│  code: c["contract_id"].upper()       │
│  name: "process_contract"             │
└───────────────────────────────────────┘
         ▲
         │
 process_contract  ←── label in local scope
 transform         ←── another label pointing to the same object
 pipeline[0]       ←── a third label
```

`process_contract`, `transform`, and `pipeline[0]` all point to the same function object. You can pass it around, return it, store it in a list — it is just a value with `()`.

**Why this matters for GenAI:** Retry wrappers, validation callbacks, prompt transforms, and middleware chains all pass functions as arguments. Every decorator you write in Module 2.2 relies on this.

```python
# Passing a function as an argument
def apply(fn, contract):
    return fn(contract)

result = apply(process_contract, {"contract_id": "nda-001", "firm": "Brennan & Carson"})
# result == "NDA-001"

# Storing functions in a data structure
validators = [extract_parties, tag_contract, cache_result]
for validate in validators:
    validate(contract)

# Returning a function from a function
def make_tagger(prefix):
    def tag(contract):
        return f"{prefix}:{contract['contract_id']}"
    return tag   # returns the inner function — a closure

bc_tagger = make_tagger("BC")
bc_tagger({"contract_id": "NDA-001"})   # "BC:NDA-001"
```

**The Java gotcha:** Java engineers think functions must be methods on a class. Python functions are standalone objects. The strategy pattern, the command pattern, the decorator pattern — all of these map to first-class functions in Python without a wrapper class.

> ⚠️ **Common mistake (Java engineers):** Writing a single-method class to pass behaviour. In Python:
> ```python
> # Java-brained: unnecessary class just to pass a function
> class ContractProcessor:
>     def process(self, c): return c["contract_id"].upper()
>
> # Pythonic: just pass the function
> def process_contract(c): return c["contract_id"].upper()
> apply(process_contract, contract)
> ```

> **My prediction — Part 1.A:**
>
> ```python
> def apply(fn, value):
>     return fn(value)
>
> result = apply(str.upper, "contract_nda")
> print(result)
> print(type(str.upper))
> ```
>
> What does `result` contain? What type is `str.upper`?
>
> My prediction: ___________

In [ ]:
def apply(fn, value):
    return fn(value)

result = apply(str.upper, "contract_nda")
print(f"result: {result!r}")
print(f"type(str.upper): {type(str.upper)}")
print()
print("str.upper is a method descriptor — a callable object.")
print("apply() calls it with 'contract_nda' as the self argument.")
print("Functions, methods, and lambdas are all callable objects — first-class values.")

> 🗂️ **Sticky phrase — Part 1:** "Functions are objects. Pass them, return them, store them — they're just values with `()`."

---

### Exercise 1 — Implementation: `compose`

Marcus uses a `compose` utility to chain contract text transforms (clean whitespace, strip legal boilerplate, normalise firm names) into a single function.

**Task:** Write `compose(f, g)` that returns a new function which applies `g` first, then `f`. Use it to compose a normalisation pipeline.

<details>
<summary>Hint (expand only after a genuine attempt)</summary>

`compose(f, g)` must return a function (define an inner function inside `compose` and return it). The returned function takes one argument `x` and returns `f(g(x))` — `g` is applied first, then `f` to the result. Think about what compose means mathematically: `(f ∘ g)(x) = f(g(x))`.

</details>

In [ ]:
def compose(f, g):
    """Return a function that applies g then f: compose(f, g)(x) == f(g(x))."""
    # TODO: implement
    ...


# ── Verify block ──────────────────────────────────────────────────────────────
def strip_whitespace(text: str) -> str:
    return " ".join(text.split())

def normalise_firm(text: str) -> str:
    return text.replace("B&C", "Brennan & Carson")

try:
    clean = compose(normalise_firm, strip_whitespace)
    assert callable(clean), "compose must return a callable"
    print("Requirement 1: PASSED — compose returns a callable")

    result = clean("  B&C  contract text  ")
    assert result == "Brennan & Carson contract text", f"Got: {result!r}"
    print("Requirement 2: PASSED — g applied first (strip), then f (normalise)")

    # Verify order: g is applied BEFORE f
    upper_first = compose(str.upper, str.strip)
    assert upper_first("  hello  ") == "HELLO", f"Got: {upper_first('  hello  ')!r}"
    print("Requirement 3: PASSED — application order is g then f")

    print("Score: 3/3")
except NotImplementedError:
    print("Not implemented — write your solution in compose()")
except AssertionError as e:
    print(f"FAILED: {e}")
except TypeError:
    print("compose() returned None — did you forget to return the inner function?")

---
## Part 2 — `*args`, `**kwargs`, keyword-only, positional-only

These are the signatures that make decorators and wrappers possible.

```python
# *args: collects extra positional arguments into a tuple
def log_contracts(*contracts):
    for c in contracts:   # contracts is a tuple
        print(c["contract_id"])

log_contracts(nda, msa, sow)   # works with any number

# **kwargs: collects extra keyword arguments into a dict
def tag_contract(**metadata):
    print(metadata)   # {'firm': 'Brennan & Carson', 'status': 'pending'}

tag_contract(firm="Brennan & Carson", status="pending")

# Combined: the decorator signature — passes everything through
def wrapper(*args, **kwargs):
    return original_fn(*args, **kwargs)   # unpack at the call site

# Keyword-only (after bare *): caller MUST use the keyword
def process_contract(contract, *, validate=True, cache=False):
    pass

process_contract(nda, validate=False)    # OK
process_contract(nda, False)             # TypeError — validate is keyword-only

# Positional-only (before /): caller may NOT use the keyword
def extract_id(contract, /):
    return contract["contract_id"]

extract_id(nda)                  # OK
extract_id(contract=nda)         # TypeError — positional-only
```

**Why this matters for GenAI:** The retry wrapper (`make_retry`) in today's final exercise uses `*args, **kwargs` to be transparent — it passes through whatever arguments the wrapped function expects without knowing them.

**The JS gotcha:** JavaScript has `...rest` at the definition site and spread `...args` at the call site. Python's `*args`/`**kwargs` is the same idea but works for both positional and keyword arguments separately.

> ⚠️ **Common mistake:** Using `*args` when the function should have explicit, typed parameters. `*args` is for true pass-through (decorators, wrappers). If you know what arguments you need, name them. Variadic signatures hide information from type checkers and from readers.

> **My prediction — Part 2.A:**
>
> ```python
> def f(*args, **kwargs):
>     return args, kwargs
>
> result = f(1, 2, firm="Brennan & Carson", status="pending")
> print(result)
> ```
>
> What does `result` contain? What are the types of `args` and `kwargs`?
>
> My prediction: ___________

In [ ]:
def f(*args, **kwargs):
    return args, kwargs

result = f(1, 2, firm="Brennan & Carson", status="pending")
print(f"result: {result}")
print(f"args type: {type(result[0])}  — tuple, always")
print(f"kwargs type: {type(result[1])}  — dict, always")
print()
print("Unpacking at call site:")
my_args = ("NDA-001", "MSA-002")
my_kwargs = {"firm": "Brennan & Carson", "status": "pending"}
print(f(*my_args, **my_kwargs))

> 🗂️ **Sticky phrase — Part 2:** "`*args` collects positional extras into a tuple. `**kwargs` collects keyword extras into a dict. Both at definition AND call site."

---

### Exercise 2 — Implementation: `format_messages`

Priya's LLM client needs a function that formats an arbitrary number of message dicts for the Anthropic API, optionally prepending a system prompt.

**Task:** Write `format_messages(*messages, system_prompt=None)` that returns a list of messages, with the system prompt prepended as `{"role": "system", "content": system_prompt}` if provided.

<details>
<summary>Hint (expand only after a genuine attempt)</summary>

`system_prompt=None` after `*messages` makes it keyword-only (the bare `*messages` creates the keyword-only boundary). Start with `result = []`. If `system_prompt is not None`, append the system dict first. Then extend with `messages` (which is a tuple). Return `result`.

</details>

In [ ]:
def format_messages(*messages, system_prompt=None):
    """
    Takes any number of message dicts.
    Prepends system_prompt as {"role": "system", "content": ...} if provided.
    Returns a list of message dicts.
    """
    # TODO: implement
    ...


# ── Verify block ──────────────────────────────────────────────────────────────
user_msg = {"role": "user", "content": "Summarise this NDA."}
asst_msg = {"role": "assistant", "content": "The NDA covers..."}

try:
    # Without system prompt
    result = format_messages(user_msg, asst_msg)
    assert result == [user_msg, asst_msg], f"No system: {result}"
    print("Requirement 1: PASSED — messages returned without system prompt")

    # With system prompt
    result2 = format_messages(user_msg, asst_msg, system_prompt="You are a contract analyst.")
    assert len(result2) == 3, f"Expected 3 messages, got {len(result2)}"
    assert result2[0] == {"role": "system", "content": "You are a contract analyst."}
    assert result2[1] == user_msg
    print("Requirement 2: PASSED — system prompt prepended correctly")

    # system_prompt is keyword-only
    try:
        format_messages(user_msg, "not a dict")  # positional args go into messages
    except Exception:
        pass
    result3 = format_messages(system_prompt="You are a contract analyst.")
    assert result3 == [{"role": "system", "content": "You are a contract analyst."}]
    print("Requirement 3: PASSED — system_prompt is keyword-only, works with zero messages")

    print("Score: 3/3")
except NotImplementedError:
    print("Not implemented — write your solution in format_messages()")
except AssertionError as e:
    print(f"FAILED: {e}")
except TypeError:
    print("format_messages() returned None — did you forget the return statement?")

---
## Consolidation pause 1 — after Parts 1 and 2

Step away and answer these without looking at the notebook.

**Q1.** `apply(len, "Brennan & Carson")` — what does it return and why?

**Q2.** When is `**kwargs` the right tool vs explicit named parameters?

**Q3.** Recall the Module 1.1 sticky phrase: "`=` moves a ___. It never ___ an object."

**Q4.** Recall the Part 1 sticky phrase: "Functions are objects. Pass them, ___ them, ___ them — they're just values with ___."

---

<details>
<summary>Expected answers</summary>

**A1.** `16` — `len` is a callable object; `apply` calls it with `"Brennan & Carson"` as the argument.

**A2.** `**kwargs` is for true pass-through — decorators and wrappers that must forward all keyword arguments without knowing them in advance. If you know what keyword arguments the function needs, name them explicitly. Explicit is always better for readability and type checking.

**A3.** "`=` moves a **label**. It never **copies** an object."

**A4.** "Functions are objects. Pass them, **return** them, **store** them — they're just values with **`()`**."

</details>

---
## Part 3 — Closures: the real definition

A closure is not magic. It is a function that captures a reference to variables in its enclosing scope — those variables live on inside the closure object even after the outer function's stack frame is gone.

**The mechanism, step by step:**

```
def make_counter():
    count = 0                     ← count lives in make_counter's local scope

    def increment():
        nonlocal count
        count += 1
        return count

    return increment              ← returns the inner function object


counter = make_counter()          ← make_counter() finishes and returns
                                     make_counter's stack frame is GONE
                                     BUT count still exists — inside counter

counter()  →  1
counter()  →  2
counter()  →  3

counter.__closure__[0].cell_contents  →  3  (the live count variable)
```

**Why the variable outlives the stack frame:** Python's closure mechanism stores captured variables in "cell objects" attached to the function. They are not on the stack — they are on the heap, and they live as long as the closure function object exists.

**The late-binding trap — the most common closure bug:**

```python
# JavaScript: var has the same issue; let was added to fix it
# Python: no equivalent fix — you must use the default argument trick

# WRONG: all lambdas close over the SAME variable i
# When called, i has already reached its final value (4)
functions = [lambda: i for i in range(5)]
print([f() for f in functions])   # [4, 4, 4, 4, 4]  — NOT [0, 1, 2, 3, 4]

# RIGHT: capture the CURRENT VALUE of i using a default argument
# Default arguments are evaluated at definition time, not call time
functions = [lambda i=i: i for i in range(5)]
print([f() for f in functions])   # [0, 1, 2, 3, 4]
```

**Why this matters for GenAI:** Rate limiters, retry wrappers, token budget enforcers, memoization — all closures that maintain state across calls. This is the most-used pattern for stateful utilities in the Anthrolytics codebase.

> ⚠️ **Common mistake:** Assuming the closure "freezes" the value at definition time. It captures the **variable** (the label), not the value. If the variable changes later, the closure sees the new value at call time.

> **My prediction — Part 3.A — the late-binding trap:**
>
> ```python
> functions = [lambda: i for i in range(5)]
> print([f() for f in functions])
> ```
>
> What does this print? Explain why in one sentence.
>
> My prediction: ___________

In [ ]:
# The late-binding trap
functions = [lambda: i for i in range(5)]
print("Late binding (wrong):", [f() for f in functions])
print("  Why: all 5 lambdas close over the SAME variable 'i'.")
print("  After the loop, i == 4. Every lambda returns 4.")
print()

# The fix: default argument snapshots the current value
functions_fixed = [lambda i=i: i for i in range(5)]
print("Default arg fix (correct):", [f() for f in functions_fixed])
print("  Why: the default argument i=i is evaluated at definition time.")
print("  Each lambda gets its own copy of the value at that point in the loop.")
print()
print("Alternative fix: use a factory function")
def make_fn(n):
    return lambda: n   # n is a separate local variable in each call
functions_factory = [make_fn(i) for i in range(5)]
print("Factory fix:", [f() for f in functions_factory])

> 🗂️ **Sticky phrase — Part 3:** "A closure captures the variable, not the value. If the variable changes, the closure sees the change — unless you snapshot it."

---

### Exercise 3 — Implementation: `make_token_bucket` (structure)

Marcus wants to see the closure structure before you add the timing logic. Start with the counter skeleton — same pattern as `make_counter`, but returning a `check()` function that raises an error when tokens are exhausted.

**Task:** Write `make_token_bucket(burst: int)` — a closure that:
- Maintains `tokens_remaining` (starts at `burst`)
- Returns a `check(tokens_needed=1)` function
- `check()` deducts `tokens_needed` from the bucket and returns `tokens_remaining`
- `check()` raises `ValueError("bucket empty")` when `tokens_remaining < tokens_needed`

<details>
<summary>Hint (expand only after a genuine attempt)</summary>

Define `tokens_remaining = burst` inside `make_token_bucket`. Define `check(tokens_needed=1)` as an inner function that uses `nonlocal tokens_remaining` to modify the enclosing variable. Check if `tokens_remaining >= tokens_needed` before deducting. Return `check`.

</details>

In [ ]:
def make_token_bucket(burst: int):
    """
    Returns a check(tokens_needed=1) function.
    check() deducts tokens and returns remaining count.
    Raises ValueError('bucket empty') when insufficient tokens.
    """
    # TODO: implement this closure
    ...


# ── Verify block ──────────────────────────────────────────────────────────────
try:
    check = make_token_bucket(burst=5)
    assert callable(check), "make_token_bucket must return a callable"
    print("Requirement 1: PASSED — returns a callable")

    r1 = check()
    r2 = check()
    r3 = check(tokens_needed=2)
    assert r1 == 4, f"After 1 token: expected 4, got {r1}"
    assert r2 == 3, f"After 2 tokens: expected 3, got {r2}"
    assert r3 == 1, f"After 2 more tokens: expected 1, got {r3}"
    print("Requirement 2: PASSED — token deduction correct (1 + 1 + 2 = 4 from burst=5)")

    try:
        check(tokens_needed=2)   # only 1 remaining
        assert False, "Should have raised ValueError"
    except ValueError as e:
        assert "bucket empty" in str(e).lower() or "empty" in str(e).lower()
    print("Requirement 3: PASSED — ValueError raised when bucket insufficient")

    # Two independent buckets
    b1 = make_token_bucket(burst=3)
    b2 = make_token_bucket(burst=3)
    b1()
    assert b1() == 1  # b1 used 2
    assert b2() == 2  # b2 only used 1 — independent
    print("Requirement 4: PASSED — independent closures have independent state")

    print("Score: 4/4")
except NotImplementedError:
    print("Not implemented — write your closure in make_token_bucket()")
except AssertionError as e:
    print(f"FAILED: {e}")
except TypeError:
    print("make_token_bucket() returned None — did you forget to return check?")

---
## Part 4 — The `nonlocal` keyword

`nonlocal` is the keyword that makes closures with mutable state work. Without it, assigning to an enclosing variable creates a *new local variable* instead — and Python raises `UnboundLocalError` because it tries to read the local variable before it is assigned.

**The exact failure:**

```
def make_counter():
    count = 0

    def increment():                 ← Python analyses this function at compile time
        count += 1                   ← count appears on the LEFT of assignment
                                       Python marks count as LOCAL to increment
        return count                 ← reads count BEFORE it was assigned locally
                                       UnboundLocalError: referenced before assignment

    def increment_fixed():
        nonlocal count               ← tells Python: count is in the ENCLOSING scope
        count += 1                   ← now modifies the enclosing count
        return count

    return increment_fixed
```

**Why the error message is confusing:** `UnboundLocalError: local variable 'count' referenced before assignment` makes it sound like count doesn't exist. It does exist — in the enclosing scope. But Python has marked `count` as local to `increment` because of the assignment on that line. The fix is `nonlocal` — it tells Python which scope to look in.

**When you need `nonlocal`:** Any closure that *mutates* an enclosing variable (`+=`, `-=`, `=` reassignment). Reading an enclosing variable does not require `nonlocal` — only mutation does.

```python
# Reading: no nonlocal needed
def make_tagger(prefix):
    def tag(contract_id):
        return f"{prefix}:{contract_id}"   # just reads prefix
    return tag

# Mutating: nonlocal required
def make_counter():
    count = 0
    def increment():
        nonlocal count               # required because count += 1 is mutation
        count += 1
        return count
    return increment
```

> ⚠️ **Common mistake:** Forgetting `nonlocal` in any closure that mutates state. The symptom is `UnboundLocalError` on the mutation line — which is confusing because the variable clearly exists in the outer function.

> **My prediction — Part 4.A:**
>
> ```python
> def make_broken_counter():
>     count = 0
>     def increment():
>         count += 1       # no nonlocal
>         return count
>     return increment
>
> counter = make_broken_counter()
> counter()   # what happens?
> ```
>
> Which line raises an error? What error? Why?
>
> My prediction: ___________

In [ ]:
def make_broken_counter():
    count = 0
    def increment():
        count += 1      # no nonlocal
        return count
    return increment

counter = make_broken_counter()
try:
    counter()
except UnboundLocalError as e:
    print(f"UnboundLocalError: {e}")
    print()
    print("Why: Python analyses increment() at compile time.")
    print("'count += 1' is an assignment → Python marks 'count' as LOCAL to increment.")
    print("When increment() runs, it tries to READ 'count' (to do count + 1).")
    print("But 'count' is local and has not been assigned yet. UnboundLocalError.")
    print()
    print("The fix:")

def make_fixed_counter():
    count = 0
    def increment():
        nonlocal count   # tell Python: count is in the ENCLOSING scope
        count += 1
        return count
    return increment

counter2 = make_fixed_counter()
print(f"  counter2(): {counter2()}")
print(f"  counter2(): {counter2()}")
print(f"  counter2(): {counter2()}")

> 🗂️ **Sticky phrase — Part 4:** "Without `nonlocal`, `count += 1` creates a new local. With it, it modifies the enclosing one."

---

### Exercise 4 — Implementation: fix the broken rate limiter

Priya found a `UnboundLocalError` in Jordan's prototype rate limiter. Fix it.

**Task:** The function below raises `UnboundLocalError`. Add the missing `nonlocal` declaration and verify it works.

<details>
<summary>Hint (expand only after a genuine attempt)</summary>

Identify which variable inside the inner function is being assigned (mutated). That variable needs `nonlocal` declared at the top of the inner function. There may be more than one.

</details>

In [ ]:
def make_rate_limiter_broken(max_calls: int):
    """Rate limiter: allow max_calls total, then raise RuntimeError."""
    calls_made = 0

    def check_rate():
        # TODO: add nonlocal declaration(s) here
        calls_made += 1              # FAILS without nonlocal
        if calls_made > max_calls:
            raise RuntimeError(f"Rate limit exceeded: {calls_made}/{max_calls} calls")
        return calls_made

    return check_rate


# ── Verify block ──────────────────────────────────────────────────────────────
try:
    limiter = make_rate_limiter_broken(max_calls=3)

    r1 = limiter()
    r2 = limiter()
    r3 = limiter()
    assert r1 == 1 and r2 == 2 and r3 == 3, f"Expected 1, 2, 3 — got {r1}, {r2}, {r3}"
    print("Requirement 1: PASSED — 3 calls within limit succeed")

    try:
        limiter()   # 4th call — should raise
        assert False, "Should have raised RuntimeError on call 4"
    except RuntimeError:
        pass
    print("Requirement 2: PASSED — RuntimeError raised on call 4 (over limit)")

    print("Score: 2/2")
except UnboundLocalError as e:
    print(f"UnboundLocalError still present — add the missing nonlocal declaration: {e}")
except AssertionError as e:
    print(f"FAILED: {e}")

---
## Consolidation pause 2 — after Parts 3 and 4

Step away and answer these without looking.

**Q1.** What does a closure capture — the value of the variable or the variable itself?

**Q2.** When does a closure raise `UnboundLocalError` on a line like `count += 1`?

**Q3.** Draw (on paper) the closure structure for `make_token_bucket(burst=5)`: the outer function, the inner function, the captured variable, and what happens after `make_token_bucket` returns.

**Q4.** Recall the Part 3 sticky phrase: "A closure captures the ___, not the ___. If the variable ___, the closure sees the change — unless you ___ it."

---

<details>
<summary>Expected answers</summary>

**A1.** The variable itself (the label). The closure holds a reference to a cell object that contains the variable's current value. If the variable is reassigned later, the closure sees the new value.

**A2.** When `count += 1` appears inside the inner function and `nonlocal count` is absent. Python sees the assignment and marks `count` as local to the inner function. When the inner function runs, it tries to read `count` before assigning it locally — `UnboundLocalError`.

**A3.** `make_token_bucket(5)` finishes and its stack frame is gone. `tokens_remaining` (value: 5) lives on in a cell object attached to the `check` function. Each call to `check()` reads and writes through that cell.

**A4.** "A closure captures the **variable**, not the **value**. If the variable **changes**, the closure sees the change — unless you **snapshot** it."

</details>

---
## Part 5 — Lambdas: when yes, when no

A lambda is a single-expression anonymous function. Nothing more.

```python
# OK: sort key — single expression, obvious meaning
contracts.sort(key=lambda c: c["firm"])

# OK: sorted with tie-breaking — still one expression, readable
sorted_contracts = sorted(contracts, key=lambda c: (c["firm"], c["contract_id"]))

# OK: filter predicate — obvious inline
pending = filter(lambda c: c["status"] == "pending", contracts)
```

**When lambdas are wrong:**

```python
# WRONG: complex logic needs a name
extract = lambda c: c.get("metadata", {}).get("parties", [{}])[0].get("name", "unknown")

# RIGHT: give it a name
def extract_primary_party(contract):
    return contract.get("metadata", {}).get("parties", [{}])[0].get("name", "unknown")

# WRONG: assigning lambda to a variable (just write def)
process = lambda c: c["contract_id"].upper()

# RIGHT:
def process_contract(c):
    return c["contract_id"].upper()
```

**The rule:** A lambda is appropriate when it is a single expression, fits on one line at its call site, and is obvious without a name. If you need to assign it to a variable, write a `def` instead — `def` gives it a name, makes it traceable in stack traces, and is cleaner.

> ⚠️ **Common mistake:** Using a lambda to avoid writing a two-line `def`. The "brevity" cost is readability and debuggability. Lambda tracebacks show `<lambda>` — not a useful function name. `def` is almost always clearer.

> 🗂️ **Sticky phrase — Part 5:** "Lambdas for one-liners only. If you need a name to understand it, give it a name."

---

### Exercise 5 — Predict and classify: good lambda vs. bad lambda

> **My prediction — Part 5.A:**
>
> Classify each lambda: **Good use** (appropriate) or **Should be a named function**.
>
> ```python
> # A
> contracts.sort(key=lambda c: c["date"])
>
> # B
> validate = lambda c: c.get("firm") and c.get("status") and len(c.get("clauses", [])) > 0
>
> # C
> total_clauses = sum(lambda c: len(c["clauses"]) for c in contracts)
>
> # D
> firms = set(map(lambda c: c["firm"], contracts))
>
> # E
> process = lambda c, *, validate=True: extract_parties(c) if validate else c
> ```
>
> My prediction: A=___, B=___, C=___, D=___, E=___

In [ ]:
print("Classifications:")
print()
print("A: contracts.sort(key=lambda c: c['date'])")
print("   GOOD — single expression sort key, obvious meaning, fits on one line")
print()
print("B: validate = lambda c: c.get('firm') and c.get('status') and len(...) > 0")
print("   SHOULD BE NAMED FUNCTION — assigned to a variable, logic has 3 conditions.")
print("   Use: def is_valid_contract(c): ...")
print()
print("C: sum(lambda c: len(c['clauses']) for c in contracts)")
print("   SYNTAX ERROR — sum() takes an iterable, not a lambda followed by 'for'.")
print("   Correct: sum(len(c['clauses']) for c in contracts)  (generator expression)")
print()
print("D: firms = set(map(lambda c: c['firm'], contracts))")
print("   ACCEPTABLE but prefer: firms = {c['firm'] for c in contracts}  (set comprehension)")
print("   The lambda+map pattern is more JS-brained; the comprehension is idiomatic Python")
print()
print("E: process = lambda c, *, validate=True: extract_parties(c) if validate else c")
print("   SHOULD BE NAMED FUNCTION — assigned to variable, keyword-only arg, conditional.")
print("   This is a function that deserves a name and a docstring.")

---
## Part 6 — Modules, packages, and imports

Every Anthrolytics module is a Python package. Import discipline is what makes the codebase maintainable at 10 engineers.

### The package structure

```
anthrolytics/               ← top-level package
  __init__.py               ← marks directory as a package; controls exports
  ingestion/
    __init__.py
    streaming_reader.py     ← from Module 1.2
    chunker.py              ← from Module 1.2
  utils/
    __init__.py
    rate_limiter.py         ← today's deliverable
    retry.py                ← today's deliverable
  clients/
    __init__.py
```

### What `__init__.py` actually does

An empty `__init__.py` marks the directory as a package. A populated one re-exports selected names:

```python
# anthrolytics/utils/__init__.py
from anthrolytics.utils.rate_limiter import make_token_bucket
from anthrolytics.utils.retry import make_retry

# Now users can write:
from anthrolytics.utils import make_token_bucket   # cleaner import path
```

### Absolute vs relative imports

```python
# ABSOLUTE (always safe — use this)
from anthrolytics.utils.rate_limiter import make_token_bucket
from anthrolytics.ingestion.streaming_reader import read_jsonl

# RELATIVE (shorter but breaks when run as a script)
from ..utils.rate_limiter import make_token_bucket   # only works inside the package
```

**Why relative imports break:** When you run `python ingestion/streaming_reader.py` directly, Python does not know the package context. The `..` in `from ..utils` has no anchor. `ModuleNotFoundError`. Always install the package with `uv pip install -e .` and import absolutely.

### `pyproject.toml` — the only config file you need

```toml
[project]
name = "anthrolytics"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = []

[project.scripts]
jsonl-filter = "anthrolytics.tools.jsonl_filter.cli:main"

[tool.ruff]
line-length = 88
select = ["E", "F", "I"]

[tool.mypy]
strict = true
```

No `setup.py`. No `setup.cfg`. `pyproject.toml` is the modern standard (PEP 517/518).

### `uv` — the fast package manager

```bash
uv venv                     # create .venv in current directory
source .venv/bin/activate   # activate
uv pip install -e .         # install this package in editable mode
# Now: python -c "from anthrolytics.utils import make_token_bucket" works
```

`uv` is 10–100× faster than `pip` for installs. Editable mode (`-e .`) means changes to your source files take effect immediately without reinstalling.

> ⚠️ **Common mistake:** Running `python utils/rate_limiter.py` directly from the project root. This breaks relative imports and produces `ModuleNotFoundError`. Always install the package editably and import it as `from anthrolytics.utils.rate_limiter import ...`.

> **My prediction — Part 6.A:**
>
> ```python
> # Inside anthrolytics/utils/rate_limiter.py:
> from ..ingestion.streaming_reader import read_jsonl
> ```
>
> Under what condition does this import succeed? Under what condition does it fail?
>
> My prediction: ___________

In [ ]:
print("Relative import: from ..ingestion.streaming_reader import read_jsonl")
print()
print("SUCCEEDS when:")
print("  The package is installed (uv pip install -e .)")
print("  AND rate_limiter.py is imported as part of the anthrolytics package")
print("  e.g.: from anthrolytics.utils import rate_limiter")
print()
print("FAILS when:")
print("  python utils/rate_limiter.py  -- run directly as a script")
print("  Python has no package context; '..' has no anchor")
print("  ImportError: attempted relative import with no known parent package")
print()
print("The safe alternative (always works):")
print("  from anthrolytics.ingestion.streaming_reader import read_jsonl")
print("  (absolute import — no ambiguity about which package context we're in)")

> 🗂️ **Sticky phrase — Part 6:** "Packages need `__init__.py`. Imports need to be absolute. `pyproject.toml` is the only config file you need."

---

### Exercise 6 — Predict: which import fails and why?

<details>
<summary>Hint (expand only after a genuine attempt)</summary>

Think about the execution context: is the module being run as a script (via `python path/to/file.py`) or imported as part of an installed package? Relative imports only work in the second case.

</details>

In [ ]:
# Demonstrating import mechanisms (no actual imports — just the analysis)

scenarios = [
    {
        "scenario": "A: python anthrolytics/utils/rate_limiter.py  (run as script)",
        "import": "from ..ingestion.streaming_reader import read_jsonl",
        "result": "FAILS",
        "reason": "No package context. '..' has no anchor. ImportError.",
    },
    {
        "scenario": "B: from anthrolytics.utils import rate_limiter  (installed package)",
        "import": "from ..ingestion.streaming_reader import read_jsonl",
        "result": "SUCCEEDS",
        "reason": "Package context exists. '..' resolves to anthrolytics.",
    },
    {
        "scenario": "C: python anthrolytics/utils/rate_limiter.py  (run as script)",
        "import": "from anthrolytics.ingestion.streaming_reader import read_jsonl",
        "result": "SUCCEEDS if package installed",
        "reason": "Absolute import; no relative ambiguity.",
    },
]

for s in scenarios:
    print(f"Scenario {s['scenario']}")
    print(f"  Import: {s['import']}")
    print(f"  Result: {s['result']}")
    print(f"  Why:    {s['reason']}")
    print()

---
## Consolidation pause 3 — after Parts 5 and 6

Step away and answer these without looking.

**Q1.** Name 3 situations where a lambda is appropriate and 2 where it is not.

**Q2.** What does an empty `__init__.py` do? What does a populated one do differently?

**Q3.** Why does running a module directly break relative imports?

**Q4.** Recall the Part 3 sticky phrase: "A closure captures the ___, not the ___."

---

<details>
<summary>Expected answers</summary>

**A1.** Appropriate: (1) single-expression sort key, (2) inline predicate in `filter()`, (3) trivial transform in `map()`. Not appropriate: (1) assigned to a named variable — use `def` instead, (2) logic spanning more than one expression or requiring a conditional.

**A2.** Empty `__init__.py`: marks the directory as a Python package, nothing more. Populated: also re-exports selected names, so importers can use shorter paths.

**A3.** Relative imports (`from ..utils import x`) require a package context — the interpreter needs to know which package the module belongs to. Running `python module.py` directly sets `__name__ == '__main__'` and `__package__ == None`. No package context → no anchor for `..` → `ImportError`.

**A4.** "A closure captures the **variable**, not the **value**."

</details>

---
## Final exercise — Marcus's 4 PM demo

**Slack — Marcus Okafor → you, 3:50 PM**

> "Ready. I'll run the stress test first — the rate limiter needs to hold the burst correctly. Then I'll install the package from a clean virtualenv and verify the import path. Score 5/5 and I'll mark week 1 ready for Priya's checkpoint review on Friday."

---

### Worked example — the `make_counter` template

Before writing `make_token_bucket`, here is the structural pattern. Every stateful closure follows this exact shape:

<details>
<summary>Reference closure pattern (study before implementing)</summary>

```python
def make_counter(start: int = 0):
    """Returns an increment() function that counts from start."""
    count = start              # ← enclosing state variable

    def increment(by: int = 1) -> int:
        nonlocal count         # ← required for mutation
        count += by            # ← mutates enclosing variable
        return count

    return increment           # ← return the inner function, not its result


# Usage
counter = make_counter(start=0)
print(counter())    # 1
print(counter(5))   # 6
print(counter())    # 7

# Two independent closures
c1 = make_counter()
c2 = make_counter()
c1()   # 1  (c1's count)
c2()   # 1  (c2's count — independent)
```

Key points:
1. State lives in the enclosing function's local variables
2. `nonlocal` is required for any mutation
3. `return increment` returns the function object — not `increment()` (calling it)
4. Each call to `make_counter()` creates a new, independent closure with its own `count`

</details>

---

Now implement both utilities below, then run Marcus's demo script.

In [ ]:
import time


class RateLimitError(Exception):
    """Raised when the token bucket is empty."""
    def __init__(self, message: str, retry_after: float = 0.0):
        super().__init__(message)
        self.retry_after = retry_after   # seconds until bucket refills


class MaxRetriesExceeded(Exception):
    """Raised when all retry attempts are exhausted."""
    pass


# ── PR 1: utils/rate_limiter.py ───────────────────────────────────────────────
def make_token_bucket(rate_per_second: float, burst: int):
    """
    Closure-based token bucket rate limiter.

    Returns check(tokens_needed=1) that:
    - Refills tokens based on elapsed time since last check
    - Deducts tokens_needed from the bucket
    - Raises RateLimitError(retry_after=...) when bucket has insufficient tokens
    """
    # TODO: implement this closure
    # State needed: tokens (float, starts at burst), last_refill_time
    # Refill logic: tokens += (now - last_refill_time) * rate_per_second
    # Cap tokens at burst
    ...


# ── PR 2: utils/retry.py ──────────────────────────────────────────────────────
def make_retry(max_attempts: int, base_delay: float, retryable: tuple):
    """
    Closure-based retry wrapper factory.

    Returns with_retry(fn) that:
    - Calls fn(*args, **kwargs)
    - On exception in retryable: waits base_delay * (2 ** attempt) seconds and retries
    - After max_attempts failures: raises MaxRetriesExceeded
    - On success: returns fn's return value
    """
    # TODO: implement this closure
    ...

In [ ]:
# ── Marcus's demo script ──────────────────────────────────────────────────────
import time

score = 0
total = 5

print("Marcus's 4 PM demo")
print("=" * 44)

# Test 1: make_token_bucket returns a callable
try:
    check = make_token_bucket(rate_per_second=10, burst=5)
    assert callable(check), "make_token_bucket must return a callable"
    print("Test 1: PASSED — make_token_bucket returns a callable")
    score += 1
except (AssertionError, NotImplementedError, TypeError) as e:
    print(f"Test 1: FAILED — {e}")

# Test 2: Burst of 5 allows 5 rapid calls
try:
    bucket = make_token_bucket(rate_per_second=10, burst=5)
    for _ in range(5):
        bucket()
    print("Test 2: PASSED — burst=5 allows 5 rapid calls")
    score += 1
except (RateLimitError, NotImplementedError, TypeError) as e:
    print(f"Test 2: FAILED — burst should allow 5 calls: {e}")

# Test 3: 6th call raises RateLimitError
try:
    bucket2 = make_token_bucket(rate_per_second=10, burst=5)
    for _ in range(5):
        bucket2()
    try:
        bucket2()   # 6th call — should raise
        print("Test 3: FAILED — 6th call should raise RateLimitError")
    except RateLimitError as e:
        assert hasattr(e, 'retry_after'), "RateLimitError must have retry_after attribute"
        print(f"Test 3: PASSED — 6th call raises RateLimitError(retry_after={e.retry_after:.3f}s)")
        score += 1
except (NotImplementedError, TypeError) as e:
    print(f"Test 3: FAILED — {e}")

# Test 4: make_retry returns a with_retry wrapper
try:
    retry = make_retry(max_attempts=3, base_delay=0.01, retryable=(ValueError,))
    assert callable(retry), "make_retry must return a callable"
    print("Test 4: PASSED — make_retry returns a callable")
    score += 1
except (AssertionError, NotImplementedError, TypeError) as e:
    print(f"Test 4: FAILED — {e}")

# Test 5: with_retry retries on retryable exceptions, raises MaxRetriesExceeded
try:
    call_count = [0]
    def flaky_extract():
        call_count[0] += 1
        if call_count[0] < 3:
            raise ValueError("transient error")
        return "NDA-001"  # succeeds on 3rd attempt

    retry_wrapper = make_retry(max_attempts=3, base_delay=0.001, retryable=(ValueError,))
    result = retry_wrapper(flaky_extract)
    assert result == "NDA-001", f"Expected 'NDA-001', got {result!r}"
    assert call_count[0] == 3, f"Expected 3 calls (2 retries), got {call_count[0]}"
    print(f"Test 5: PASSED — retried {call_count[0]-1} time(s), succeeded on attempt {call_count[0]}")
    score += 1
except (NotImplementedError, TypeError, AssertionError) as e:
    print(f"Test 5: FAILED — {e}")

print()
print(f"Score: {score}/{total}")

if score == total:
    print()
    print("Marcus: Package installs clean. Rate limiter holds the burst correctly.")
    print("Retry logic looks solid. One thing: your RateLimitError should expose")
    print("retry_after as an attribute, not just a message string. Fix that and push.")
    print("Then you're done with week 1 theory — Priya will do the checkpoint review Friday.")
else:
    print()
    print(f"Marcus: {total - score} test(s) failing. Fix before end of day.")

---
## Module summary — screenshot this

| Concept | Sticky phrase | Quick test |
|---|---|---|
| First-class functions | "Functions are objects. Pass them, return them, store them — they're just values with `()`." | What does `apply(str.upper, "hello")` return? |
| `*args` / `**kwargs` | "`*args` collects positional extras into a tuple. `**kwargs` collects keyword extras into a dict. Both at definition AND call site." | What type is `args` inside `def f(*args): ...`? |
| Closures | "A closure captures the variable, not the value. If the variable changes, the closure sees the change — unless you snapshot it." | `[lambda: i for i in range(5)]` — what does each lambda return? |
| `nonlocal` | "Without `nonlocal`, `count += 1` creates a new local. With it, it modifies the enclosing one." | What error occurs without `nonlocal` when mutating an enclosing variable? |
| Lambdas | "Lambdas for one-liners only. If you need a name to understand it, give it a name." | When should you write `def` instead of a lambda? |
| Packages | "Packages need `__init__.py`. Imports need to be absolute. `pyproject.toml` is the only config file you need." | Why does `from ..utils import x` fail when run as a script? |

---
## What comes next — Week 1 Checkpoint: `jsonl-filter` CLI

The rate limiter and retry wrapper built today are the production utilities that wrap every LLM call in the checkpoint CLI. You now have all three pieces from Week 1:

- **Module 1.1** (mental model): explains why closures capture variables not values, why mutable defaults are a trap, and why the object model matters
- **Module 1.2** (generators): `read_jsonl` — the source of the streaming pipeline
- **Module 1.3** (closures + packaging): `make_token_bucket`, `make_retry` — the utilities that wrap every API call

The Week 1 Checkpoint (`01_01_CK`) synthesises all three. Priya's brief: build `jsonl-filter` — a CLI tool that filters contract JSONL files using a generator pipeline, handles interrupts cleanly, and ships as a `uv`-installable package. You have everything you need.

**Module 2.1 (OOP — next section):** The closure-based `make_token_bucket` pattern is one way to maintain state. Module 2.1 introduces the class-based alternative — a `TokenBucket` class. You will implement both and understand the tradeoffs: closures are simpler and have no inheritance overhead; classes are inspectable, extensible, and testable with `isinstance`. The choice between them is the first real architectural judgment call of the programme.